# Model Routing and LLM Gateways

**Phase 07** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-07/09-model-routing-llm-gateways.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

Your team ships an AI feature and defaults to `claude-sonnet-4-5` for every request because it performs best in evals. Six weeks later, your monthly AI spend is $12,000 and the finance team wants a meeting.

You look at the logs. The traffic breaks down like this: 68% of requests are single-turn Q&A over a knowledge base, average 400 input tokens. 22% are summarization tasks with 2,000-4,000 tokens. 10% are multi-hop reasoning tasks that actually need a capable model.

You were paying Sonnet prices for Haiku work on 90% of your traffic.

The fix is routing: send each request to the cheapest model that can handle it correctly. But "cheapest model that can handle it" requires a decision functi...

## The Concept

### The Routing Decision

Every incoming request has properties you can observe before calling any model: prompt length, an explicit complexity flag from the caller, and the caller's cost budget. Those three signals drive the routing decision.



The routing logic reads left to right in priority. Explicit complexity overrides everything. Then token count catches large-context tasks that need a more capable model. Then budget forces the cheap model. Finally, medium-length prompts with no budget constraint go to Sonnet.

### What an LLM Gateway Adds

A raw `anthropic.Anthropic()` client talks to...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Model Router - Phase 07, Lesson 09
Routes LLM requests to the cheapest model that can handle the task correctly.

Usage:
    python main.py

No API calls are made in this demo. The router evaluates rules and returns
(model_id, reason) tuples. Integrate with your LLM client of choice.
"""

from __future__ import annotations

import dataclasses
from typing import Literal


ModelId = Literal[
    "claude-3-5-haiku-20241022",
    "claude-sonnet-4-5",
]

# Approximate cost per token (USD) for input tokens only.
# Update these when Anthropic adjusts pricing.
MODEL_COSTS: dict[ModelId, float] = {
    "claude-3-5-haiku-20241022": 0.000001,   # $1 / 1M tokens
    "claude-sonnet-4-5":        0.000003,   # $3 / 1M tokens
}


@dataclasses.dataclass
class RoutingRule:
    name: str
    description: str

    def matches(
        self,
        prompt_tokens: int,
        complexity: str | None,
        cost_budget: float | None,
    ) -> tuple[bool, ModelId, str]:
        """Return (matched, model_id, reason)."""
        raise NotImplementedError


@dataclasses.dataclass
class ExplicitComplexityRule(RoutingRule):
    """If the caller says complexity=high, use Sonnet."""

    def matches(self, prompt_tokens, complexity, cost_budget):
        if complexity == "high":
            return True, "claude-sonnet-4-5", "explicit_complexity_high"
        return False, "claude-3-5-haiku-20241022", ""


@dataclasses.dataclass
class LargeContextRule(RoutingRule):
    """Prompts over 6,000 tokens need Sonnet for reliable recall."""
    token_threshold: int = 6_000

    def matches(self, prompt_tokens, complexity, cost_budget):
        if prompt_tokens > self.token_threshold:
            return True, "claude-sonnet-4-5", "large_context"
        return False, "claude-3-5-haiku-20241022", ""


@dataclasses.dataclass
class BudgetConstraintRule(RoutingRule):
    """If the caller has a tight cost budget, use Haiku."""
    haiku_cost_per_token: float = MODEL_COSTS["claude-3-5-haiku-20241022"]

    def matches(self, prompt_tokens, complexity, cost_budget):
        if cost_budget is None:
            return False, "claude-3-5-haiku-20241022", ""
        # Estimate cost for this prompt on Haiku
        estimated_haiku_cost = prompt_tokens * self.haiku_cost_per_token
        # If even Haiku exceeds budget, still use Haiku (caller handles the error)
        if cost_budget < estimated_haiku_cost * 3:
            return True, "claude-3-5-haiku-20241022", "budget_constraint"
        return False, "claude-3-5-haiku-20241022", ""


@dataclasses.dataclass
class MediumPromptRule(RoutingRule):
    """Prompts over 1,500 tokens without a budget constraint go to Sonnet."""
    token_threshold: int = 1_500

    def matches(self, prompt_tokens, complexity, cost_budget):
        if prompt_tokens > self.token_threshold:
            return True, "claude-sonnet-4-5", "medium_to_large_prompt"
        return False, "claude-3-5-haiku-20241022", ""


class ModelRouter:
    """
    Routes LLM requests to the cheapest model that can handle the task.

    Rules are evaluated in order. The first matching rule wins.
    If no rule matches, the default_model is used.

    Args:
        default_budget: Maximum cost (USD) per request before forcing Haiku.
            Set to None to disable budget enforcement as a default.
        default_model: Model to use when no rule matches.
    """

    def __init__(
        self,
        default_budget: float | None = None,
        default_model: ModelId = "claude-3-5-haiku-20241022",
    ):
        self.default_budget = default_budget
        self.default_model = default_model
        self.rules: list[RoutingRule] = [
            ExplicitComplexityRule(
                name="explicit_complexity",
                description="Caller explicitly flags complexity=high",
            ),
            LargeContextRule(
                name="large_context",
                description="Prompts over 6000 tokens use Sonnet",
                token_threshold=6_000,
            ),
            BudgetConstraintRule(
                name="budget_constraint",
                description="Tight cost budget forces Haiku",
            ),
            MediumPromptRule(
                name="medium_prompt",
                description="Prompts 1500-6000 tokens use Sonnet",
                token_threshold=1_500,
            ),
        ]
        print(f"ModelRouter initialized with {len(self.rules)} routing rules")

    def _estimate_tokens(self, prompt: str) -> int:
        """Rough token estimate: 4 characters per token."""
        return max(1, len(prompt) // 4)

    def route(
        self,
        prompt: str,
        complexity: str | None = None,
        cost_budget: float | None = None,
    ) -> tuple[ModelId, str]:
        """
        Route a request to the appropriate model.

        Args:
            prompt: The full prompt text (system + user combined).
            complexity: Optional hint from the caller. Pass "high" for tasks
                that require multi-hop reasoning or long-form synthesis.
            cost_budget: Maximum acceptable cost (USD) for this single call.
                If None, uses the router's default_budget.

        Returns:
            (model_id, reason) - model to call, and why this model was chosen.
        """
        effective_budget = cost_budget if cost_budget is not None else self.default_budget
        prompt_tokens = self._estimate_tokens(prompt)

        for rule in self.rules:
            matched, model, reason = rule.matches(prompt_tokens, complexity, effective_budget)
            if matched:
                return model, reason

        return self.default_model, "default_no_rule_matched"

    def estimate_cost(self, model: ModelId, prompt_tokens: int, output_tokens: int = 150) -> float:
        """Estimate the cost of a call in USD."""
        cost_per_token = MODEL_COSTS[model]
        return (prompt_tokens + output_tokens) * cost_per_token


def main():
    router = ModelRouter(default_budget=0.01)

    test_cases = [
        {
            "label": "Simple Q&A",
            "prompt": "What is the capital of France?",
            "complexity": None,
            "cost_budget": 0.001,
        },
        {
            "label": "High-complexity task",
            "prompt": "Analyze the strategic implications of this document in detail.",
            "complexity": "high",
            "cost_budget": None,
        },
        {
            "label": "Large context",
            "prompt": "x" * 7_000,  # ~7000 chars = ~1750 tokens
            "complexity": None,
            "cost_budget": None,
        },
        {
            "label": "Budget constrained",
            "prompt": "Summarize this paragraph.",
            "complexity": None,
            "cost_budget": 0.001,
        },
    ]

    total_routed_cost = 0.0
    total_unrouted_cost = 0.0

    for tc in test_cases:
        prompt_tokens = router._estimate_tokens(tc["prompt"])
        model, reason = router.route(
            prompt=tc["prompt"],
            complexity=tc.get("complexity"),
            cost_budget=tc.get("cost_budget"),
        )
        cost = router.estimate_cost(model, prompt_tokens)
        unrouted_cost = router.estimate_cost("claude-sonnet-4-5", prompt_tokens)

        total_routed_cost += cost
        total_unrouted_cost += unrouted_cost

        print(f"\nRouting test: {tc['label']}")
        print(f"  Prompt tokens (est): {prompt_tokens}")
        print(f"  Routed to: {model} ({reason})")
        print(f"  Estimated cost: ${cost:.6f}")

    savings_pct = (1 - total_routed_cost / total_unrouted_cost) * 100
    print(f"\nCost analysis:")
    print(f"  Without routing (all sonnet): ${total_unrouted_cost:.6f}")
    print(f"  With routing:                  ${total_routed_cost:.6f}")
    print(f"  Savings: {savings_pct:.1f}%")

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. What is the correct engineering decision and why?**

- A. Keep Sonnet for all traffic. A 0.02 quality gap is too risky for production.
- B. Route the 72% Q&A traffic to Haiku. A 0.02 quality difference is within noise on a 0-1 scale, and the cost reduction is 67% on that traffic segment.
- C. Run both models in parallel on all requests and return whichever finishes first.
- D. Switch all traffic to Haiku to maximize savings.

**2. Which model does the router select and why?**

- A. claude-sonnet-4-5, because no budget constraint was specified.
- B. claude-3-5-haiku-20241022, because the default model is Haiku when no rule matches.
- C. claude-sonnet-4-5, because the medium_prompt rule fires at 800 tokens.
- D. The router raises an error because complexity is None.

**3. Under what condition does the LLM classifier approach justify its cost?**

- A. Always - LLM classification is always more accurate than rules.
- B. Never - the overhead always exceeds the benefit.
- C. When the classifier demonstrably reduces misrouting enough to recover the classifier cost through better model selection, measured in a controlled A/B test.
- D. When prompt length alone is not a reliable signal, which it never is.

**4. What is the root cause and correct fix?**

- A. The explicit_complexity rule is broken and should be removed.
- B. The client library defaulted complexity='high', bypassing all other routing logic. Fix the default to complexity=None and add a lint rule preventing complexity='high' without a comment justifying it.
- C. Raise the token threshold for large_context so more traffic falls through to that rule.
- D. Remove the explicit_complexity rule entirely since it is being abused.

**5. What is the most accurate assessment of the tradeoff?**

- A. The colleague is right. LiteLLM adds no value when you use only one provider.
- B. LiteLLM is always necessary and the colleague is wrong.
- C. For a single-provider setup, direct SDK is simpler now. LiteLLM earns its place when you need provider fallbacks, normalized error handling, or cost logging across providers - deferring adds a future migration cost.
- D. LiteLLM is required for rate-limit handling, which the Anthropic SDK cannot do.

**6. What does this reveal about the routing design, and what should you do?**

- A. The router is broken. Large-context requests should always go to Haiku.
- B. The default_budget is a per-call estimate, not a hard cap. The large_context rule correctly identified the need for Sonnet. Log the overage, add a cost alert for calls exceeding 3x the budget estimate, and review whether 8000-token prompts are expected or a bug in the caller.
- C. Add a hard cap in the Anthropic client that terminates calls exceeding $0.005.
- D. Remove the large_context rule so all overlong prompts fall through to the budget_constraint rule.

**7. What is the monthly cost savings from routing?**

- A. $156
- B. $195
- C. $175
- D. $143

**8. What is the correct product and engineering response?**

- A. Remove the budget_constraint rule - it is causing quality problems.
- B. The routing decision was correct given the budget. The product response is to show the user that complex queries require the paid tier. The engineering response is to add a quality check: if the response is below a length or quality threshold, surface a 'upgrade for better answers' prompt.
- C. Override the budget for all users and absorb the extra cost.
- D. Switch all free-tier users permanently to Haiku regardless of task complexity.

_Answers are in `checks.json` in the lesson directory._